In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
with open("statute_corpus_v0 (4).json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} chunks")
print(corpus[0])   # sanity check — look at one chunk

Loaded 13 chunks
{'chunk_id': 'CONSUMER_001', 'category': 'Consumer', 'act': 'Consumer Protection Act, 2019', 'section': 'Section 35', 'section_title': 'Manner in which complaint shall be made', 'text': 'A consumer complaint about goods sold/delivered (or agreed to be) or a service provided (or agreed to be), or an allegation of unfair trade practice, may be filed with the District Commission. It may be filed by the affected consumer, a recognised consumer association (whether or not the consumer is a member), one or more consumers representing others with the same interest (with District Commission permission), or the Central/State Government. Complaints may be filed electronically and must include the prescribed fee.', 'source_url': 'https://legalbots.in/blog/how-to-file-a-consumer-complaint-in-india'}


In [4]:
# Combine title + body text — titles carry strong keyword signal
corpus_texts = [f"{c['section_title']} {c['text']}" for c in corpus]

vectorizer = TfidfVectorizer(stop_words="english")
corpus_vectors = vectorizer.fit_transform(corpus_texts)

print("Vector shape:", corpus_vectors.shape)   # (13, vocab_size)

Vector shape: (13, 339)


In [5]:
def search_corpus(query, predicted_category=None, top_k=3, min_score=0.05):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, corpus_vectors)[0]
    ranked = sorted(zip(corpus, scores), key=lambda x: x[1], reverse=True)

    if predicted_category:
        # Category is already trusted from the classifier — just scope to it,
        # no threshold needed since there are only 1-3 chunks per category anyway
        category_matches = [c for c, s in ranked if c["category"] == predicted_category]
        results = category_matches[:top_k]
    else:
        # No category given — fall back to threshold-filtered global search
        results = [c for c, s in ranked if s >= min_score][:top_k]

    return results, ranked

In [6]:
query = "I bought a defective product and the seller refuses to help"

results, ranked = search_corpus(query)   # no category yet

print("All scores:")
for chunk, score in ranked:
    print(f"  {chunk['chunk_id']}: {score:.3f}")

print("\nTop results (no category filter):")
for r in results:
    print(f"{r['chunk_id']} | {r['category']} | {r['act']} - {r['section']}")

All scores:
  FAMILY_001: 0.120
  CONSUMER_001: 0.000
  CONSUMER_002: 0.000
  EMPLOYMENT_001: 0.000
  EMPLOYMENT_002: 0.000
  FAMILY_002: 0.000
  WOMENSAFETY_001: 0.000
  WOMENSAFETY_002: 0.000
  PROPERTY_001: 0.000
  PROPERTY_002: 0.000
  LEGALAID_001: 0.000
  LEGALAID_002: 0.000
  OTHER_001: 0.000

Top results (no category filter):
FAMILY_001 | Family | Bharatiya Nagarik Suraksha Sanhita (BNSS), 2023 - Section 144 (formerly Section 125, CrPC 1973, effective till 30 June 2024)


In [7]:
# For now, pass the category manually to test the fix.
# Later, replace "Consumer" with your trained classifier's prediction:
#   predicted_category = classifier.predict([query])[0]
predicted_category = "Consumer"

results, _ = search_corpus(query, predicted_category=predicted_category)

print(f"Predicted category: {predicted_category}\n")
for r in results:
    print("Chunk ID:", r["chunk_id"])
    print("Category:", r["category"])
    print("Act:", r["act"])
    print("Section:", r["section"])
    print("Title:", r["section_title"])
    print()

Predicted category: Consumer

Chunk ID: CONSUMER_001
Category: Consumer
Act: Consumer Protection Act, 2019
Section: Section 35
Title: Manner in which complaint shall be made

Chunk ID: CONSUMER_002
Category: Consumer
Act: Consumer Protection Act, 2019
Section: Sections 34, 47, 58 (as revised by 2021 Rules)
Title: Pecuniary jurisdiction of consumer forums



In [8]:
test_cases = [
    ("My landlord won't return my rental deposit", "Property"),
    ("My husband refuses to give me and my kids money to live", "Family"),
    ("My employer hasn't paid my salary for 2 months", "Employment"),
]

for q, cat in test_cases:
    results, _ = search_corpus(q, predicted_category=cat)
    print(f"Query: {q}")
    print(f"Category: {cat}")
    for r in results:
        print(f"  → {r['chunk_id']} | {r['act']} - {r['section']}")
    print()

Query: My landlord won't return my rental deposit
Category: Property
  → PROPERTY_001 | Specific Relief Act, 1963 - General (recovery of possession, declaratory and injunctive relief)
  → PROPERTY_002 | Karnataka Land Revenue Act, 1964 - Sections 128, 131 (mutation) and Section 104 (unauthorised occupation)

Query: My husband refuses to give me and my kids money to live
Category: Family
  → FAMILY_001 | Bharatiya Nagarik Suraksha Sanhita (BNSS), 2023 - Section 144 (formerly Section 125, CrPC 1973, effective till 30 June 2024)
  → FAMILY_002 | Family Courts Act, 1984 - General

Query: My employer hasn't paid my salary for 2 months
Category: Employment
  → EMPLOYMENT_001 | Industrial Relations Code, 2020 - Section 59 (formerly Section 33C, Industrial Disputes Act 1947 - repealed 21 Nov 2025)
  → EMPLOYMENT_002 | Minimum Wages Act, 1948 - Section 20



In [9]:
import pandas as pd

df = pd.read_excel("Dataset_1_finalized (1).xlsx", sheet_name="Legal_Problems")
print(df.shape)
print(df.columns.tolist())
df.head()

(134, 12)
['case_id', 'category', 'subcategory', 'example_user_problem', 'risk_level', 'possible_pathway', 'documents_or_evidence', 'possible_authority', 'source_family', 'safety_note', 'legal_citation', 'disclaimer']


,case_id,category,subcategory,example_user_problem,risk_level,possible_pathway,documents_or_evidence,possible_authority,source_family,safety_note,legal_citation,disclaimer
0,1,Employment,Unpaid wages,My employer has not paid my wages for the last...,Medium,Labour/employment grievance pathway; verify th...,Employment records; salary slips; bank stateme...,Relevant DLSA/TLSC or competent labour authori...,India Code / NALSA / relevant labour authority,Triage only; verify current procedure and juri...,"Payment of Wages Act, 1936, Section 15 (claim ...","This is legal-information triage, not legal ad..."
1,2,Employment,Unpaid wages,My company keeps delaying my pending salary.,Medium,Labour/employment grievance pathway; verify th...,Employment records; salary slips; bank stateme...,Relevant DLSA/TLSC or competent labour authori...,India Code / NALSA / relevant labour authority,Triage only.,"Payment of Wages Act, 1936, Section 15 (claim ...","This is legal-information triage, not legal ad..."
2,3,Employment,Unpaid wages,I have not received my salary for two months a...,Medium,Labour/employment grievance pathway; verify th...,Employment records; salary slips; bank stateme...,Relevant DLSA/TLSC or competent labour authori...,India Code / NALSA / relevant labour authority,Triage only; verify current procedure and juri...,"Payment of Wages Act, 1936, Section 15 (claim ...","This is legal-information triage, not legal ad..."
3,4,Employment,Unpaid wages,My factory owner has withheld my daily wages f...,Medium,Labour/employment grievance pathway; verify th...,Employment records; salary slips; bank stateme...,Relevant DLSA/TLSC or competent labour authori...,India Code / NALSA / relevant labour authority,Triage only.,"Payment of Wages Act, 1936, Section 15 (claim ...","This is legal-information triage, not legal ad..."
4,5,Employment,Unpaid wages,My contractor is refusing to pay the wages owe...,Medium,Labour/employment grievance pathway; verify th...,Employment records; salary slips; bank stateme...,Relevant DLSA/TLSC or competent labour authori...,India Code / NALSA / relevant labour authority,Triage only; verify current procedure and juri...,"Payment of Wages Act, 1936, Section 15 (claim ...","This is legal-information triage, not legal ad..."


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

text_col = "example_user_problem"      # <-- replace with your real column name
category_col = "category"     # <-- replace with your real column name

X = df[text_col]
y = df[category_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf_vectorizer = TfidfVectorizer(stop_words="english")
X_train_vec = clf_vectorizer.fit_transform(X_train)
X_test_vec = clf_vectorizer.transform(X_test)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_vec, y_train)

y_pred = classifier.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6296296296296297


In [11]:
def classify_category(query):
    query_vec = clf_vectorizer.transform([query])
    predicted = classifier.predict(query_vec)[0]
    probabilities = classifier.predict_proba(query_vec)[0]
    confidence = max(probabilities)
    return predicted, confidence

# quick test
cat, conf = classify_category("I bought a defective product and the seller refuses to help")
print(f"Predicted: {cat} (confidence: {conf:.2f})")

Predicted: Consumer (confidence: 0.40)


In [12]:
def triage_query(query, min_confidence=0.3):
    category, confidence = classify_category(query)
    
    if confidence < min_confidence:
        return {
            "query": query,
            "category": None,
            "confidence": confidence,
            "message": "Low confidence — could not confidently classify this query.",
            "results": []
        }
    
    results, _ = search_corpus(query, predicted_category=category)
    
    return {
        "query": query,
        "category": category,
        "confidence": confidence,
        "results": results
    }

In [13]:
output = triage_query("I bought a defective product and the seller refuses to help")

print("Query:", output["query"])
print("Predicted category:", output["category"])
print("Confidence:", f"{output['confidence']:.2f}")
print()
for r in output["results"]:
    print(f"{r['chunk_id']} | {r['act']} - {r['section']}")
    print(f"  {r['section_title']}")

Query: I bought a defective product and the seller refuses to help
Predicted category: Consumer
Confidence: 0.40

CONSUMER_001 | Consumer Protection Act, 2019 - Section 35
  Manner in which complaint shall be made
CONSUMER_002 | Consumer Protection Act, 2019 - Sections 34, 47, 58 (as revised by 2021 Rules)
  Pecuniary jurisdiction of consumer forums


In [14]:
results, ranked = search_corpus(
    "I bought a defective product and the seller refuses to help",
    predicted_category="Consumer"
)
print("Results:", results)
print()
print("All scores for Consumer-relevant check:")
for chunk, score in ranked:
    print(f"  {chunk['chunk_id']} ({chunk['category']}): {score:.3f}")

Results: [{'chunk_id': 'CONSUMER_001', 'category': 'Consumer', 'act': 'Consumer Protection Act, 2019', 'section': 'Section 35', 'section_title': 'Manner in which complaint shall be made', 'text': 'A consumer complaint about goods sold/delivered (or agreed to be) or a service provided (or agreed to be), or an allegation of unfair trade practice, may be filed with the District Commission. It may be filed by the affected consumer, a recognised consumer association (whether or not the consumer is a member), one or more consumers representing others with the same interest (with District Commission permission), or the Central/State Government. Complaints may be filed electronically and must include the prescribed fee.', 'source_url': 'https://legalbots.in/blog/how-to-file-a-consumer-complaint-in-india'}, {'chunk_id': 'CONSUMER_002', 'category': 'Consumer', 'act': 'Consumer Protection Act, 2019', 'section': 'Sections 34, 47, 58 (as revised by 2021 Rules)', 'section_title': 'Pecuniary jurisdic